In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [2]:
import pandas as pd

# Load data
train_df = pd.read_csv(
    "/kaggle/input/datasets/ngcnhtnguyn/nlp-midterm-dataset/train.csv",
    header=None
)

valid_df = pd.read_csv(
    "/kaggle/input/datasets/ngcnhtnguyn/nlp-midterm-dataset/valid.csv",
    header=None
)

test_df = pd.read_csv(
    "/kaggle/input/datasets/ngcnhtnguyn/nlp-midterm-dataset/test.csv",
    header=None
)

# Set column names
columns = ["label", "text"]

train_df.columns = columns
valid_df.columns = columns
test_df.columns = columns


In [3]:
train_df = train_df[train_df["label"] != "label"]
valid_df = valid_df[valid_df["label"] != "label"]
test_df  = test_df[test_df["label"] != "label"]

In [4]:
label_list = sorted(train_df["label"].unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}

train_df["label"] = train_df["label"].map(label2id)
test_df["label"]   = test_df["label"].map(label2id)
valid_df["label"] = valid_df["label"].map(label2id)

In [5]:
print(label2id)

{'Irrelevant': 0, 'Negative': 1, 'Neutral': 2, 'Positive': 3}


In [6]:
# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
from datasets import Dataset

# ---- Ensure string type ----
for df in [train_df, valid_df, test_df]:
    df["text"] = df["text"].astype(str)

# ---- Convert to HF Dataset ----
train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_df[["text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]]
)

# ---- Tokenize ----
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

# ---- Set torch format ----
cols = ["input_ids", "attention_mask", "label"]

train_dataset.set_format(type="torch", columns=cols)
valid_dataset.set_format(type="torch", columns=cols)
test_dataset.set_format(type="torch", columns=cols)

Map:   0%|          | 0/66595 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/7400 [00:00<?, ? examples/s]

In [8]:
# Model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id)
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    report = classification_report(
        labels,
        preds,
        target_names=list(label2id.keys()),
        output_dict=True
    )

    return {
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
    }

In [10]:
# Training args
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
    eval_dataset=valid_dataset, 
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
# Train
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
500,2.157697
1000,1.690802
1500,1.409378
2000,1.117885
2500,0.662638
3000,0.540577
3500,0.489550
4000,0.457345
4500,0.328677
5000,0.252796


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6246, training_loss=0.7732513206645613, metrics={'train_runtime': 2791.0425, 'train_samples_per_second': 71.581, 'train_steps_per_second': 2.238, 'total_flos': 1.314164653161984e+16, 'train_loss': 0.7732513206645613, 'epoch': 3.0})

In [12]:
# Evaluate
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.2566179633140564, 'eval_precision': 0.9731925593326792, 'eval_recall': 0.972972972972973, 'eval_f1': 0.9729568347031371, 'eval_runtime': 5.0538, 'eval_samples_per_second': 197.671, 'eval_steps_per_second': 6.332, 'epoch': 3.0}


In [13]:
from sklearn.metrics import classification_report

preds = trainer.predict(test_dataset)
y_pred = preds.predictions.argmax(axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=label2id.keys()))

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

  Irrelevant       0.93      0.91      0.92      1288
    Negative       0.95      0.93      0.94      2236
     Neutral       0.89      0.94      0.91      1811
    Positive       0.93      0.91      0.92      2065

    accuracy                           0.92      7400
   macro avg       0.92      0.92      0.92      7400
weighted avg       0.93      0.92      0.92      7400

